# ETL — WNM excitatory: Cell Features (three feature sets)

Writes WNM excitatory neuron cell features across three feature sets: **`exc_visp_morph_features`** (shared with `etl_visp_exc_patchseq_02`; defs/set owned by that notebook), **`wnm_exc_local_axon_features`** (local axon + apical dendrite morphology), and **`wnm_exc_complete_axon_features`** (whole-brain axon features from fMOST — placeholder, file not yet available). All rows use `project_id="visp_wnm"`. Prerequisites: `etl_wnm_exc_01_dataset_dataitem.ipynb` and `etl_visp_exc_patchseq_02_cell_features.ipynb`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.io.arrow_utils import (
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    DataItem,
    DataItemDataSetAssociation,
)
from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models


In [2]:
WIDE_CSV_SET1  = "/data/visp-features-and-mapping/RawFeaturesWide_ChamferCorr.csv"
WIDE_CSV_SET2  = "/data/exc_vis_manuscript_wnm_axon_projection/AxonRawReatureWide.csv"
OUTPUT_ROOT    = get_settings().output_root
PROJECT_ID     = "visp_wnm"
DATASET_ID     = "visp_exc_wnm"   # prereq assertion only
FSI_SHARED     = "exc_visp_morph_features"
FSI_LOCAL      = "wnm_exc_local_axon_features"
FSI_FMOST      = "wnm_exc_complete_axon_features"

print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"PROJECT_ID  : {PROJECT_ID}")
print(f"DATASET_ID  : {DATASET_ID}")
print(f"FSI_SHARED  : {FSI_SHARED}")
print(f"FSI_LOCAL   : {FSI_LOCAL}")
print(f"FSI_FMOST   : {FSI_FMOST}")

OUTPUT_ROOT : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID  : visp_wnm
DATASET_ID  : visp_exc_wnm
FSI_SHARED  : exc_visp_morph_features
FSI_LOCAL   : wnm_exc_local_axon_features
FSI_FMOST   : wnm_exc_complete_axon_features


## Prerequisite checks

In [3]:
# Assert WNM DataItems registered via association table.
assoc = (
    pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association")
    .filter(
        (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
    )
)
assert assoc.shape[0] > 0, (
    f"etl_wnm_exc_01 must be run first — "
    f"no DataItemDataSetAssociation rows for dataset_id='{DATASET_ID}'"
)
wnm_registered_ids = set(assoc["dataitem_id"].to_list())
print(f"WNM DataItems registered: {len(wnm_registered_ids)}")

# Assert the shared CellFeatureSet exists (written by etl_visp_exc_patchseq_02).
cfs_check = pl.read_delta(OUTPUT_ROOT / "cellfeatureset").filter(pl.col("id") == FSI_SHARED)
assert cfs_check.shape[0] == 1, (
    f"etl_visp_exc_patchseq_02 must be run first — "
    f"CellFeatureSet '{FSI_SHARED}' not found"
)
print(f"Shared CellFeatureSet '{FSI_SHARED}' found.")

WNM DataItems registered: 341
Shared CellFeatureSet 'exc_visp_morph_features' found.


---
## Set 1 — `exc_visp_morph_features` (shared defs; WNM rows only)

Defs and `CellFeatureSet` are owned by `etl_visp_exc_patchseq_02_cell_features.ipynb`. This notebook only writes WNM rows to `cellfeatures/exc_visp_morph_features/` and the corresponding `CellFeatureMatrix` pointer.

In [4]:
# Load the shared feature defs written by exc patchseq _02.
shared_defs_df = (
    pl.read_delta(OUTPUT_ROOT / "cellfeaturedefinition")
    .filter(pl.col("feature_set_id") == FSI_SHARED)
    .sort("id")
)
print("Shared defs shape:", shared_defs_df.shape)
assert shared_defs_df.shape[0] > 0, f"No defs found for feature_set_id='{FSI_SHARED}'"
shared_def_ids = shared_defs_df["id"].to_list()
print("Feature IDs (first 5):", shared_def_ids[:5])

Shared defs shape: (50, 8)
Feature IDs (first 5): ['apical_dendrite_bias_x', 'apical_dendrite_bias_y', 'apical_dendrite_depth_pc_0', 'apical_dendrite_depth_pc_1', 'apical_dendrite_depth_pc_2']


In [5]:
# Load Set 1 wide CSV. Drop unnamed index column; id col = swc_path.
wide1_raw = pd.read_csv(WIDE_CSV_SET1)
wide1_raw = wide1_raw.drop(columns=["Unnamed: 0"], errors="ignore")
wide1_raw = wide1_raw.rename(columns={"swc_path": "id"})
print("Set1 wide CSV shape:", wide1_raw.shape)
wide1_raw.head(3)

Set1 wide CSV shape: (345, 45)


,id,soma_aligned_dist_from_pia,basal_dendrite_max_euclidean_distance,apical_dendrite_num_branches,basal_dendrite_stem_exit_down,apical_dendrite_bias_y,apical_dendrite_extent_y,apical_dendrite_depth_pc_0,apical_dendrite_early_branch_path,apical_dendrite_mean_contraction,...,apical_dendrite_extent_x,apical_dendrite_std_moments_along_max_distance_projection,apical_dendrite_num_outer_bifurcations,apical_dendrite_max_path_distance,apical_dendrite_depth_pc_1,basal_dendrite_calculate_number_of_stems,basal_dendrite_mean_contraction,apical_dendrite_bias_x,basal_dendrite_bias_x,basal_dendrite_frac_below_apical_dendrite
0,17109_6201-X4328-Y6753_reg,755.648634,153.118741,13.112095,0.519306,51.220063,130.348098,-127.799230,0.812125,0.919582,...,92.771122,0.095100,0.017473,188.477520,-4.550537,2.025048,0.924001,116.965793,73.426861,0.038531
1,17109_6301-X4756-Y24516_reg,779.803826,158.644468,11.710442,0.346235,29.822118,187.891473,-125.915996,0.464351,0.942480,...,133.943725,0.047786,0.017473,298.755658,-2.846434,3.016802,0.954847,68.235786,20.734153,-0.005611
2,17109_6601-X4384-Y7436_reg,727.831495,218.932446,18.718725,0.000094,53.307739,162.011671,-130.399448,0.573624,0.931142,...,139.756745,0.170515,0.759907,229.434422,-9.068610,2.025048,0.928544,144.969883,46.419273,0.232868


In [6]:
# Check which shared def cols are missing from this CSV and warn loudly.
csv_feat_cols = set(wide1_raw.columns) - {"id"}
missing_cols = [d for d in shared_def_ids if d not in csv_feat_cols]
extra_cols   = [c for c in csv_feat_cols if c not in shared_def_ids]

if missing_cols:
    print(f"WARNING: {len(missing_cols)} shared def columns are missing from Set1 CSV — "
          f"will be filled with NaN:\n  {missing_cols}")
    for col in missing_cols:
        wide1_raw[col] = np.nan

if extra_cols:
    print(f"WARNING: {len(extra_cols)} columns in CSV are NOT in shared defs — dropping:\n  {extra_cols}")
    wide1_raw = wide1_raw.drop(columns=extra_cols)

print(f"After alignment: {len(missing_cols)} NaN-filled, {len(extra_cols)} dropped")

  ['apical_dendrite_mean_diameter', 'apical_dendrite_total_surface_area', 'axon_exit_distance', 'axon_exit_theta', 'basal_dendrite_mean_diameter', 'basal_dendrite_total_surface_area']
After alignment: 6 NaN-filled, 0 dropped


In [7]:
# Check id coverage; register any new cells not in _01.
set1_ids = list(wide1_raw["id"].astype(str))
new_ids_set1 = [i for i in set1_ids if i not in wnm_registered_ids]
print(f"Cells in Set1 CSV       : {len(set1_ids)}")
print(f"Already in DataItem     : {len(set1_ids) - len(new_ids_set1)}")
print(f"New to register (Set 1) : {len(new_ids_set1)}")
if new_ids_set1:
    print("New ids:", new_ids_set1)

Cells in Set1 CSV       : 345
Already in DataItem     : 341
New to register (Set 1) : 4
New ids: ['17109_6801-X7432-Y4405_reg', '211541_6961-X18505-Y15909_reg', '220309_5824-X3486-Y10261_reg', '221686_5481-X4093-Y13144_reg']


In [8]:
# Register new cells (DataItem + DataItemDataSetAssociation) for those in Set1 not yet in _01.
if new_ids_set1:
    new_items = [DataItem(id=i, name=i, project_id=PROJECT_ID) for i in new_ids_set1]
    n_appended = write_models(new_items, output_root=OUTPUT_ROOT).rows_written
    print(f"Appended {n_appended} new DataItem rows")
else:
    print("All Set1 cells already registered \u2014 no new DataItem writes.")

# Re-assert the full (project_id, dataset_id) association scope as the union
# of any existing assoc rows and the Set1 ids. DataItemDataSetAssociation is
# overwrite_scoped on (project_id, dataset_id), so passing the full intended
# set is idempotent and self-heals partial prior runs.
existing_assoc = (
    pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
existing_assoc_ids = set(existing_assoc["dataitem_id"].to_list())
all_assoc_ids = sorted(existing_assoc_ids | set(set1_ids))
n_assoc = write_models(
    [DataItemDataSetAssociation(dataitem_id=i, dataset_id=DATASET_ID, project_id=PROJECT_ID)
     for i in all_assoc_ids],
    output_root=OUTPUT_ROOT,
).rows_written
print(f"Associations written for ({PROJECT_ID}, {DATASET_ID}): {n_assoc}")

# Refresh registered ids so Set2 coverage check reflects newly added cells.
wnm_registered_ids = set(
    pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
    ["dataitem_id"].to_list()
)


Appended 4 new DataItem rows
Associations written for (visp_wnm, visp_exc_wnm): 345


In [9]:
# Cast feature columns to their defined data types (from shared defs), add partition cols.
shared_defs_dict = {row["id"]: row["data_type"] for row in shared_defs_df.iter_rows(named=True)}
for feat_id, dtype_str in shared_defs_dict.items():
    wide1_raw[feat_id] = wide1_raw[feat_id].astype(np.dtype(dtype_str))

wide1_raw["project_id"]     = PROJECT_ID
wide1_raw["feature_set_id"] = FSI_SHARED

# Reorder: id first, then feature cols in def order, then partition cols.
col_order = ["id"] + shared_def_ids + ["project_id", "feature_set_id"]
wide1_df = wide1_raw[col_order]
print("Set1 wide table shape:", wide1_df.shape)

# Reconstruct CellFeatureDefinition objects for schema building.
shared_defs_objs = [
    CellFeatureDefinition(
        id=row["id"],
        data_type=row["data_type"],
        unit=row["unit"] if row["unit"] else None,
        description=row["description"] if row["description"] else None,
        project_id=row["project_id"],
        feature_set_id=row["feature_set_id"],
    )
    for row in shared_defs_df.iter_rows(named=True)
]
shared_cfs_obj = CellFeatureSet(
    id=FSI_SHARED,
    feature_definition_ids=shared_def_ids,
    project_id="visp_patchseq",  # owned by exc patchseq project
)
schema_wide1 = build_cell_feature_matrix_schema(
    shared_cfs_obj, shared_defs_objs, cell_index_column="id"
)

arrow_table1 = pa.Table.from_pandas(wide1_df, schema=schema_wide1)

Set1 wide table shape: (345, 53)


In [10]:
# Write Set1 wide-form parquet.
write_deltalake(
    OUTPUT_ROOT / f"cellfeatures/{FSI_SHARED}", arrow_table1,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FSI_SHARED}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Set1 wide-form written:", arrow_table1.shape)

Set1 wide-form written: (345, 53)


In [11]:
# Verification — Set1 wide parquet.
set1_v = pl.read_delta(OUTPUT_ROOT / f"cellfeatures/{FSI_SHARED}").filter(
    pl.col("project_id") == PROJECT_ID
)
print(set1_v.shape)
print(set1_v.select(["id", "project_id", "feature_set_id"]).head(3))
assert set1_v.shape[0] == len(set1_ids)
assert set1_v["id"].n_unique() == len(set1_ids)
assert (set1_v["project_id"] == PROJECT_ID).all()

(345, 53)
shape: (3, 3)
┌─────────────────────────────┬────────────┬─────────────────────────┐
│ id                          ┆ project_id ┆ feature_set_id          │
│ ---                         ┆ ---        ┆ ---                     │
│ str                         ┆ str        ┆ str                     │
╞═════════════════════════════╪════════════╪═════════════════════════╡
│ 17109_6201-X4328-Y6753_reg  ┆ visp_wnm   ┆ exc_visp_morph_features │
│ 17109_6301-X4756-Y24516_reg ┆ visp_wnm   ┆ exc_visp_morph_features │
│ 17109_6601-X4384-Y7436_reg  ┆ visp_wnm   ┆ exc_visp_morph_features │
└─────────────────────────────┴────────────┴─────────────────────────┘


In [12]:
# Write CellFeatureMatrix pointer for Set1.
cfm1 = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FSI_SHARED}",
    feature_set_id=FSI_SHARED,
    parquet_path=f"file://{OUTPUT_ROOT.resolve()}/cellfeatures/{FSI_SHARED}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm1], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [13]:
# Verification — CellFeatureMatrix Set1.
cfm1_v = pl.read_delta(OUTPUT_ROOT / "cellfeaturematrix").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_SHARED)
)
print(cfm1_v.shape); print(cfm1_v)
assert cfm1_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬────────────┐
│ id                  ┆ feature_set_id      ┆ parquet_path        ┆ cell_index_column ┆ project_id │
│ ---                 ┆ ---                 ┆ ---                 ┆ ---               ┆ ---        │
│ str                 ┆ str                 ┆ str                 ┆ str               ┆ str        │
╞═════════════════════╪═════════════════════╪═════════════════════╪═══════════════════╪════════════╡
│ visp_wnm_exc_visp_m ┆ exc_visp_morph_feat ┆ file:///scratch/em_ ┆ id                ┆ visp_wnm   │
│ orph_featur…        ┆ ures                ┆ patchseq_wn…        ┆                   ┆            │
└─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴────────────┘


---
## Set 2 — `wnm_exc_local_axon_features`

In [14]:
# Load Set2 wide CSV. id col = specimen_id.
wide2_raw = pd.read_csv(WIDE_CSV_SET2)
wide2_raw = wide2_raw.rename(columns={"specimen_id": "id"})
wide2_raw["id"] = wide2_raw["id"].astype(str)
print("Set2 wide CSV shape:", wide2_raw.shape)
wide2_raw.head(3)

Set2 wide CSV shape: (345, 52)


,id,apical_dendrite_bias_x,apical_dendrite_bias_y,apical_dendrite_depth_pc_0,apical_dendrite_depth_pc_1,apical_dendrite_depth_pc_2,apical_dendrite_depth_pc_3,apical_dendrite_depth_pc_4,apical_dendrite_early_branch_path,apical_dendrite_extent_x,...,axon_max_branch_order,axon_max_euclidean_distance,axon_max_path_distance,axon_mean_contraction,axon_num_branches,axon_soma_percentile_x,axon_soma_percentile_y,axon_total_length,soma_aligned_dist_from_pia,soma_surface_area
0,17109_6201-X4328-Y6753_reg,21.311538,2.306327,-446.959647,-77.960333,-116.411504,-89.641313,-76.897975,0.479785,291.778779,...,18.0,782.489366,2295.547989,0.775924,133.0,0.456671,0.291300,32679.601463,755.648634,0.0
1,17109_6301-X4756-Y24516_reg,39.591795,17.363311,-456.398402,-70.969916,-123.361715,-92.973863,-99.533545,0.484623,255.485239,...,11.0,798.003108,1794.220795,0.789746,83.0,0.434223,0.193180,24944.239274,779.803826,0.0
2,17109_6601-X4384-Y7436_reg,110.472066,-7.655321,-447.486796,-103.720271,-125.564660,-117.194281,-51.706742,0.498860,331.019359,...,10.0,722.175168,2298.330681,0.773319,79.0,0.297082,0.511213,20598.298943,727.831495,0.0


In [15]:
# Check id coverage against registered WNM DataItems.
set2_ids = list(wide2_raw["id"])
missing_set2 = set(set2_ids) - wnm_registered_ids
if missing_set2:
    print(f"WARNING: {len(missing_set2)} Set2 ids not in DataItem: {list(missing_set2)[:5]}")
else:
    print(f"All {len(set2_ids)} Set2 cell ids are registered in DataItem.")

All 345 Set2 cell ids are registered in DataItem.


In [16]:
# Build CellFeatureDefinition rows from column names. data_type = "<f8", unit/description blank.
feat_cols_2 = [c for c in wide2_raw.columns if c != "id"]
feature_defs_2 = [
    CellFeatureDefinition(
        id=col,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FSI_LOCAL,
    )
    for col in feat_cols_2
]
print(f"Built {len(feature_defs_2)} CellFeatureDefinition rows for '{FSI_LOCAL}'")

Built 51 CellFeatureDefinition rows for 'wnm_exc_local_axon_features'


In [17]:
# Write CellFeatureDefinition for Set2.
result = write_models(feature_defs_2, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition written: {result.rows_written} rows")

CellFeatureDefinition written: 51 rows


In [18]:
# Verification — CellFeatureDefinition Set2.
cfd2_v = pl.read_delta(OUTPUT_ROOT / "cellfeaturedefinition").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_LOCAL)
)
print(cfd2_v.shape); print(cfd2_v.head(3))
assert cfd2_v.shape[0] == len(feature_defs_2)
assert cfd2_v["id"].n_unique() == len(feature_defs_2)

(51, 8)
shape: (3, 8)
┌──────────────┬─────────────┬──────┬───────────┬───────────┬───────────┬────────────┬─────────────┐
│ id           ┆ description ┆ unit ┆ data_type ┆ range_min ┆ range_max ┆ project_id ┆ feature_set │
│ ---          ┆ ---         ┆ ---  ┆ ---       ┆ ---       ┆ ---       ┆ ---        ┆ _id         │
│ str          ┆ str         ┆ str  ┆ str       ┆ f64       ┆ f64       ┆ str        ┆ ---         │
│              ┆             ┆      ┆           ┆           ┆           ┆            ┆ str         │
╞══════════════╪═════════════╪══════╪═══════════╪═══════════╪═══════════╪════════════╪═════════════╡
│ apical_dendr ┆ null        ┆ null ┆ <f8       ┆ null      ┆ null      ┆ visp_wnm   ┆ wnm_exc_loc │
│ ite_bias_x   ┆             ┆      ┆           ┆           ┆           ┆            ┆ al_axon_fea │
│              ┆             ┆      ┆           ┆           ┆           ┆            ┆ tures       │
│ apical_dendr ┆ null        ┆ null ┆ <f8       ┆ null      ┆ null   

In [19]:
# Build and write CellFeatureSet for Set2.
feature_set_2 = CellFeatureSet(
    id=FSI_LOCAL,
    description="WNM excitatory neuron local axon and apical dendrite morphology features.",
    feature_definition_ids=[fd.id for fd in feature_defs_2],
    project_id=PROJECT_ID,
)
result = write_models([feature_set_2], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet written: {result.rows_written} rows")

CellFeatureSet written: 1 rows


In [20]:
# Verification — CellFeatureSet Set2.
cfs2_v = pl.read_delta(OUTPUT_ROOT / "cellfeatureset").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("id") == FSI_LOCAL)
)
print(cfs2_v.shape); print(cfs2_v)
assert cfs2_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌───────────────────────┬──────────────────┬──────────────────────┬───────────────────┬────────────┐
│ id                    ┆ description      ┆ feature_definition_i ┆ extraction_method ┆ project_id │
│ ---                   ┆ ---              ┆ ds                   ┆ ---               ┆ ---        │
│ str                   ┆ str              ┆ ---                  ┆ str               ┆ str        │
│                       ┆                  ┆ list[str]            ┆                   ┆            │
╞═══════════════════════╪══════════════════╪══════════════════════╪═══════════════════╪════════════╡
│ wnm_exc_local_axon_fe ┆ WNM excitatory   ┆ ["apical_dendrite_bi ┆ null              ┆ visp_wnm   │
│ atures                ┆ neuron local ax… ┆ as_x", "ap…          ┆                   ┆            │
└───────────────────────┴──────────────────┴──────────────────────┴───────────────────┴────────────┘


In [21]:
# Cast feature columns, add partition cols, write wide-form parquet for Set2.
for col in feat_cols_2:
    wide2_raw[col] = wide2_raw[col].astype(np.float64)

wide2_raw["project_id"]     = PROJECT_ID
wide2_raw["feature_set_id"] = FSI_LOCAL

col_order2 = ["id"] + feat_cols_2 + ["project_id", "feature_set_id"]
wide2_df = wide2_raw[col_order2]

schema_wide2 = build_cell_feature_matrix_schema(
    feature_set_2, feature_defs_2, cell_index_column="id"
)
arrow_table2 = pa.Table.from_pandas(wide2_df, schema=schema_wide2)

write_deltalake(
    OUTPUT_ROOT / f"cellfeatures/{FSI_LOCAL}", arrow_table2,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FSI_LOCAL}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Set2 wide-form written:", arrow_table2.shape)

Set2 wide-form written: (345, 54)


In [22]:
# Verification — Set2 wide parquet.
set2_v = pl.read_delta(OUTPUT_ROOT / f"cellfeatures/{FSI_LOCAL}").filter(
    pl.col("project_id") == PROJECT_ID
)
print(set2_v.shape)
print(set2_v.select(["id", "project_id", "feature_set_id"]).head(3))
assert set2_v.shape[0] == len(set2_ids)
assert set2_v["id"].n_unique() == len(set2_ids)
assert (set2_v["project_id"] == PROJECT_ID).all()

(345, 54)
shape: (3, 3)
┌─────────────────────────────┬────────────┬─────────────────────────────┐
│ id                          ┆ project_id ┆ feature_set_id              │
│ ---                         ┆ ---        ┆ ---                         │
│ str                         ┆ str        ┆ str                         │
╞═════════════════════════════╪════════════╪═════════════════════════════╡
│ 17109_6201-X4328-Y6753_reg  ┆ visp_wnm   ┆ wnm_exc_local_axon_features │
│ 17109_6301-X4756-Y24516_reg ┆ visp_wnm   ┆ wnm_exc_local_axon_features │
│ 17109_6601-X4384-Y7436_reg  ┆ visp_wnm   ┆ wnm_exc_local_axon_features │
└─────────────────────────────┴────────────┴─────────────────────────────┘


In [23]:
# Write CellFeatureMatrix pointer for Set2.
cfm2 = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FSI_LOCAL}",
    feature_set_id=FSI_LOCAL,
    parquet_path=f"file://{OUTPUT_ROOT.resolve()}/cellfeatures/{FSI_LOCAL}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm2], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [24]:
# Verification — CellFeatureMatrix Set2.
cfm2_v = pl.read_delta(OUTPUT_ROOT / "cellfeaturematrix").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_LOCAL)
)
print(cfm2_v.shape); print(cfm2_v)
assert cfm2_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬────────────┐
│ id                  ┆ feature_set_id      ┆ parquet_path        ┆ cell_index_column ┆ project_id │
│ ---                 ┆ ---                 ┆ ---                 ┆ ---               ┆ ---        │
│ str                 ┆ str                 ┆ str                 ┆ str               ┆ str        │
╞═════════════════════╪═════════════════════╪═════════════════════╪═══════════════════╪════════════╡
│ visp_wnm_wnm_exc_lo ┆ wnm_exc_local_axon_ ┆ file:///scratch/em_ ┆ id                ┆ visp_wnm   │
│ cal_axon_fe…        ┆ features            ┆ patchseq_wn…        ┆                   ┆            │
└─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴────────────┘


---
## Set 3 — `wnm_exc_complete_axon_features`

In [25]:
WIDE_CSV_SET3 = "/data/exc_vis_manuscript_wnm_axon_projection/fMOST_Complete_Axon_Features.csv"

# Load Set3 CSV. id col = Unnamed: 0; strip .swc suffix.
wide3_raw = pd.read_csv(WIDE_CSV_SET3)
wide3_raw["id"] = wide3_raw["Unnamed: 0"].str.removesuffix(".swc")
wide3_raw = wide3_raw.drop(columns=["Unnamed: 0"])
print("Set3 wide CSV shape:", wide3_raw.shape)
wide3_raw.head(3)

Set3 wide CSV shape: (341, 19)


,complete_axon_total_length,complete_axon_max_euclidean_distance,complete_axon_num_branches,complete_axon_num_tips,complete_axon_max_branch_order,complete_axon_max_path_distance,complete_axon_mean_contraction,axon_width,axon_depth,axon_height,complete_axon_total_number_of_targets,complete_axon_total_projection_length,complete_axon_VIS_length,complete_axon_ipsi_VIS_length,complete_axon_length_in_soma_structure,complete_axon_number_of_VIS_targets,complete_axon_number_of_contra_VIS_targets,fraction_of_complete_axon_in_soma_structure,id
0,77517.582963,2471.512730,199.0,100.0,19.0,2981.977200,0.772691,2190.436660,3174.806000,1894.232709,4,65541.812655,65541.812655,65541.812655,22772.756376,4,0,0.347454,17109_6201-X4328-Y6753_reg
1,79605.518533,6632.085331,231.0,116.0,25.0,9869.076728,0.787281,1795.214651,7271.248616,1411.582850,6,53599.221355,53599.221355,20993.605335,19826.395642,6,4,0.369901,17109_6301-X4756-Y24516_reg
2,86975.409840,8133.325899,221.0,111.0,14.0,12340.915374,0.799677,2459.358099,8190.705000,3460.126000,6,64073.986251,64073.986251,60850.159820,15966.887800,6,1,0.249195,17109_6601-X4384-Y7436_reg


In [26]:
# Check id coverage against registered WNM DataItems.
set3_ids = list(wide3_raw["id"])
missing_set3 = set(set3_ids) - wnm_registered_ids
if missing_set3:
    print(f"WARNING: {len(missing_set3)} Set3 ids not in DataItem: {list(missing_set3)[:5]}")
else:
    print(f"All {len(set3_ids)} Set3 cell ids are registered in DataItem.")

All 341 Set3 cell ids are registered in DataItem.


In [27]:
# Build CellFeatureDefinition rows from column names. data_type = "<f8".
feat_cols_3 = [c for c in wide3_raw.columns if c != "id"]
feature_defs_3 = [
    CellFeatureDefinition(
        id=col,
        data_type="<f8",
        project_id=PROJECT_ID,
        feature_set_id=FSI_FMOST,
    )
    for col in feat_cols_3
]
print(f"Built {len(feature_defs_3)} CellFeatureDefinition rows for '{FSI_FMOST}'")

Built 18 CellFeatureDefinition rows for 'wnm_exc_complete_axon_features'


In [28]:
# Write CellFeatureDefinition for Set3.
result = write_models(feature_defs_3, output_root=OUTPUT_ROOT)
print(f"CellFeatureDefinition written: {result.rows_written} rows")

CellFeatureDefinition written: 18 rows


In [29]:
# Verification — CellFeatureDefinition Set3.
cfd3_v = pl.read_delta(OUTPUT_ROOT / "cellfeaturedefinition").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_FMOST)
)
print(cfd3_v.shape); print(cfd3_v.head(3))
assert cfd3_v.shape[0] == len(feature_defs_3)
assert cfd3_v["id"].n_unique() == len(feature_defs_3)

(18, 8)
shape: (3, 8)
┌──────────────┬─────────────┬──────┬───────────┬───────────┬───────────┬────────────┬─────────────┐
│ id           ┆ description ┆ unit ┆ data_type ┆ range_min ┆ range_max ┆ project_id ┆ feature_set │
│ ---          ┆ ---         ┆ ---  ┆ ---       ┆ ---       ┆ ---       ┆ ---        ┆ _id         │
│ str          ┆ str         ┆ str  ┆ str       ┆ f64       ┆ f64       ┆ str        ┆ ---         │
│              ┆             ┆      ┆           ┆           ┆           ┆            ┆ str         │
╞══════════════╪═════════════╪══════╪═══════════╪═══════════╪═══════════╪════════════╪═════════════╡
│ complete_axo ┆ null        ┆ null ┆ <f8       ┆ null      ┆ null      ┆ visp_wnm   ┆ wnm_exc_com │
│ n_total_leng ┆             ┆      ┆           ┆           ┆           ┆            ┆ plete_axon_ │
│ th           ┆             ┆      ┆           ┆           ┆           ┆            ┆ features    │
│ complete_axo ┆ null        ┆ null ┆ <f8       ┆ null      ┆ null   

In [30]:
# Build and write CellFeatureSet for Set3.
feature_set_3 = CellFeatureSet(
    id=FSI_FMOST,
    description=(
        "WNM excitatory neuron whole-brain axon features from fMOST reconstructions "
        "(total length, projection length, # targets, VIS length, etc.)."
    ),
    feature_definition_ids=[fd.id for fd in feature_defs_3],
    project_id=PROJECT_ID,
)
result = write_models([feature_set_3], output_root=OUTPUT_ROOT)
print(f"CellFeatureSet written: {result.rows_written} rows")

CellFeatureSet written: 1 rows


In [31]:
# Verification — CellFeatureSet Set3.
cfs3_v = pl.read_delta(OUTPUT_ROOT / "cellfeatureset").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("id") == FSI_FMOST)
)
print(cfs3_v.shape); print(cfs3_v)
assert cfs3_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌───────────────────────┬──────────────────┬──────────────────────┬───────────────────┬────────────┐
│ id                    ┆ description      ┆ feature_definition_i ┆ extraction_method ┆ project_id │
│ ---                   ┆ ---              ┆ ds                   ┆ ---               ┆ ---        │
│ str                   ┆ str              ┆ ---                  ┆ str               ┆ str        │
│                       ┆                  ┆ list[str]            ┆                   ┆            │
╞═══════════════════════╪══════════════════╪══════════════════════╪═══════════════════╪════════════╡
│ wnm_exc_complete_axon ┆ WNM excitatory   ┆ ["complete_axon_tota ┆ null              ┆ visp_wnm   │
│ _features             ┆ neuron whole-br… ┆ l_length",…          ┆                   ┆            │
└───────────────────────┴──────────────────┴──────────────────────┴───────────────────┴────────────┘


In [32]:
# Cast feature columns to float64, add partition cols, write wide-form parquet for Set3.
for col in feat_cols_3:
    wide3_raw[col] = wide3_raw[col].astype(np.float64)

wide3_raw["project_id"]     = PROJECT_ID
wide3_raw["feature_set_id"] = FSI_FMOST

col_order3 = ["id"] + feat_cols_3 + ["project_id", "feature_set_id"]
wide3_df = wide3_raw[col_order3]

schema_wide3 = build_cell_feature_matrix_schema(
    feature_set_3, feature_defs_3, cell_index_column="id"
)
arrow_table3 = pa.Table.from_pandas(wide3_df, schema=schema_wide3)

write_deltalake(
    OUTPUT_ROOT / f"cellfeatures/{FSI_FMOST}", arrow_table3,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FSI_FMOST}'",
    partition_by=["project_id", "feature_set_id"],
)
print("Set3 wide-form written:", arrow_table3.shape)

Set3 wide-form written: (341, 21)


In [33]:
# Verification — Set3 wide parquet.
set3_v = pl.read_delta(OUTPUT_ROOT / f"cellfeatures/{FSI_FMOST}").filter(
    pl.col("project_id") == PROJECT_ID
)
print(set3_v.shape)
print(set3_v.select(["id", "project_id", "feature_set_id"]).head(3))
assert set3_v.shape[0] == len(set3_ids)
assert set3_v["id"].n_unique() == len(set3_ids)
assert (set3_v["project_id"] == PROJECT_ID).all()

(341, 21)
shape: (3, 3)
┌─────────────────────────────┬────────────┬────────────────────────────────┐
│ id                          ┆ project_id ┆ feature_set_id                 │
│ ---                         ┆ ---        ┆ ---                            │
│ str                         ┆ str        ┆ str                            │
╞═════════════════════════════╪════════════╪════════════════════════════════╡
│ 17109_6201-X4328-Y6753_reg  ┆ visp_wnm   ┆ wnm_exc_complete_axon_features │
│ 17109_6301-X4756-Y24516_reg ┆ visp_wnm   ┆ wnm_exc_complete_axon_features │
│ 17109_6601-X4384-Y7436_reg  ┆ visp_wnm   ┆ wnm_exc_complete_axon_features │
└─────────────────────────────┴────────────┴────────────────────────────────┘


In [34]:
# Write CellFeatureMatrix pointer for Set3.
cfm3 = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FSI_FMOST}",
    feature_set_id=FSI_FMOST,
    parquet_path=f"file://{OUTPUT_ROOT.resolve()}/cellfeatures/{FSI_FMOST}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
result = write_models([cfm3], output_root=OUTPUT_ROOT)
print(f"CellFeatureMatrix written: {result.rows_written} rows")

CellFeatureMatrix written: 1 rows


In [35]:
# Verification — CellFeatureMatrix Set3.
cfm3_v = pl.read_delta(OUTPUT_ROOT / "cellfeaturematrix").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FSI_FMOST)
)
print(cfm3_v.shape); print(cfm3_v)
assert cfm3_v.shape[0] == 1

(1, 5)
shape: (1, 5)
┌─────────────────────┬─────────────────────┬─────────────────────┬───────────────────┬────────────┐
│ id                  ┆ feature_set_id      ┆ parquet_path        ┆ cell_index_column ┆ project_id │
│ ---                 ┆ ---                 ┆ ---                 ┆ ---               ┆ ---        │
│ str                 ┆ str                 ┆ str                 ┆ str               ┆ str        │
╞═════════════════════╪═════════════════════╪═════════════════════╪═══════════════════╪════════════╡
│ visp_wnm_wnm_exc_co ┆ wnm_exc_complete_ax ┆ file:///scratch/em_ ┆ id                ┆ visp_wnm   │
│ mplete_axon…        ┆ on_features         ┆ patchseq_wn…        ┆                   ┆            │
└─────────────────────┴─────────────────────┴─────────────────────┴───────────────────┴────────────┘


---
## Summary

| Path | Rows written | Notes |
|---|---|---|
| `cellfeatures/exc_visp_morph_features/` (WNM rows) | 345 | Set 1; defs/set owned by `etl_visp_exc_patchseq_02` |
| `cellfeaturematrix/` (`exc_visp_morph_features`) | 1 | Set 1 pointer |
| `cellfeaturedefinition/` (`wnm_exc_local_axon_features`) | 51 | Set 2 |
| `cellfeatureset/` (`wnm_exc_local_axon_features`) | 1 | Set 2 |
| `cellfeatures/wnm_exc_local_axon_features/` | 345 | Set 2 |
| `cellfeaturematrix/` (`wnm_exc_local_axon_features`) | 1 | Set 2 |
| `cellfeaturedefinition/` (`wnm_exc_complete_axon_features`) | 18 | Set 3 |
| `cellfeatureset/` (`wnm_exc_complete_axon_features`) | 1 | Set 3 |
| `cellfeatures/wnm_exc_complete_axon_features/` | 341 | Set 3 |
| `cellfeaturematrix/` (`wnm_exc_complete_axon_features`) | 1 | Set 3 |

Set 1 defs/set not written here — owned by `etl_visp_exc_patchseq_02_cell_features.ipynb`.